# Fase 6 — Predicción final y submission

El momento de la verdad: convertir los dos finalistas de la Fase 5 en predicciones sobre los 418
pasajeros del test, subirlas a Kaggle y — por primera vez — enfrentar nuestra CV al mundo real.

## ¿Por qué reentrenar con TODO el train?

Detalle conceptual importante. Durante la validación cruzada, cada modelo se entrenaba con 4/5 de
los datos (~713 filas) — era necesario para reservar un fold como examen. Pero la CV era el
**termómetro**, no el modelo: su trabajo (estimar la calidad) ya está hecho. Para predecir el test
queremos el mejor modelo posible, y con 891 filas, regalar 178 sería absurdo. Así que:

- **Termómetro**: CV con 5 folds → nos dijo GB 0.8563 / RF 0.8473.
- **Modelo final**: mismos hiperparámetros, entrenado con el **100% del train**.

Regla de coherencia: el modelo final debe ser EXACTAMENTE el que evaluamos (mismos hiperparámetros,
mismas features) — solo cambia la cantidad de datos. Cambiar cualquier otra cosa invalidaría la
estimación de la CV.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

train = pd.read_csv("../data/processed/train_features.csv")
test  = pd.read_csv("../data/processed/test_features.csv")

X = train.drop(columns=["PassengerId", "Survived"])
y = train["Survived"]
X_test = test[X.columns]     # MISMAS columnas y MISMO orden que en el entrenamiento

finalistas = {
    "v03_gb": GradientBoostingClassifier(learning_rate=0.03, max_depth=4, n_estimators=100,
                                         subsample=1.0, random_state=42),
    "v02_rf": RandomForestClassifier(n_estimators=300, max_depth=8, max_features=0.5,
                                     min_samples_leaf=4, random_state=42),
}

predicciones = {}
for nombre, modelo in finalistas.items():
    modelo.fit(X, y)                          # 100% del train
    predicciones[nombre] = modelo.predict(X_test)
    print(f"{nombre}: tasa de supervivencia predicha en test = {predicciones[nombre].mean():.4f}")

v03_gb: tasa de supervivencia predicha en test = 0.3852


v02_rf: tasa de supervivencia predicha en test = 0.3756


## 1 · Controles de cordura sobre las predicciones

Antes de mirar el formato, miremos el **contenido**. Tres comprobaciones baratas que cazan errores
graves (features desalineadas, target invertido...):

1. **Tasa de supervivencia predicha ≈ 38%**, la proporción del train. ✔ (0.385 y 0.376 — si saliera
   0.70 o 0.10, algo estaría roto.)
2. **¿Cuánto coinciden los dos finalistas entre sí?**
3. **¿Cuánto se separan de la regla del género?** — si coincidieran al 100%, todo nuestro trabajo de
   features no habría cambiado ni una predicción; si difirieran en 200, sería sospechosísimo.

In [2]:
acuerdo = (predicciones["v03_gb"] == predicciones["v02_rf"]).mean()
print(f"acuerdo GB-RF: {acuerdo:.1%}  ({(predicciones['v03_gb'] != predicciones['v02_rf']).sum()} pasajeros de discrepancia)")

regla_genero = test["IsFemale"].values
for nombre, p in predicciones.items():
    n_diff = (p != regla_genero).sum()
    hombres_vivos = ((p == 1) & (regla_genero == 0)).sum()
    mujeres_muertas = ((p == 0) & (regla_genero == 1)).sum()
    print(f"{nombre}: difiere de la regla del genero en {n_diff} "
          f"({hombres_vivos} hombres que vivirian + {mujeres_muertas} mujeres que moririan)")

acuerdo GB-RF: 96.2%  (16 pasajeros de discrepancia)
v03_gb: difiere de la regla del genero en 51 (30 hombres que vivirian + 21 mujeres que moririan)
v02_rf: difiere de la regla del genero en 51 (28 hombres que vivirian + 23 mujeres que moririan)


Los dos modelos coinciden en el 96% del test y ambos se apartan de la regla del género en
exactamente **51 pasajeros** — hombres a los que salvan (niños `Master`, primera clase...) y mujeres
a las que condenan (3ª clase, familias grandes...). Esos 51 son, literalmente, nuestra apuesta:
donde las features de las Fases 2-3 contradicen al sexo puro. La pregunta que responderá el
leaderboard: ¿acertamos más de las que fallamos?

## 2 · Construir y validar el archivo de submission

Kaggle es estricto: **exactamente** 2 columnas (`PassengerId`, `Survived`), 418 filas + cabecera,
valores 0/1. Lo validamos con `assert`: comprobaciones que **revientan ruidosamente** si algo no
cuadra. Es infinitamente mejor que un vistazo manual — un vistazo no nota que hay 417 filas.

In [3]:
import os
os.makedirs("../submissions", exist_ok=True)

for nombre, p in predicciones.items():
    sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": p})

    assert sub.shape == (418, 2),                       "forma incorrecta"
    assert list(sub.columns) == ["PassengerId", "Survived"], "columnas mal nombradas"
    assert sub["Survived"].isin([0, 1]).all(),          "valores fuera de {0,1}"
    assert sub["PassengerId"].is_unique,                "ids duplicados"
    assert sub.isna().sum().sum() == 0,                 "hay nulos"

    ruta = f"../submissions/submission_{nombre}.csv"
    sub.to_csv(ruta, index=False)      # index=False: sin columna fantasma de numeros de fila
    print(f"OK {ruta}")

sub.head(3)

OK ../submissions/submission_v03_gb.csv
OK ../submissions/submission_v02_rf.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0


## 3 · Subir a Kaggle

Con el CLI (desde la terminal, en la carpeta del proyecto). El mensaje `-m` es importante: enlaza
la submission con la versión del registro — dentro de un mes sabrás qué era cada score:

```bash
kaggle competitions submit -c titanic -f submissions/submission_v03_gb.csv \
       -m "v03 GradientBoosting afinado (lr=0.03, depth=4, n=100) - CV 0.8563"

kaggle competitions submit -c titanic -f submissions/submission_v02_rf.csv \
       -m "v02 RandomForest afinado (depth=8, leaf=4, feat=0.5) - CV 0.8473"

kaggle competitions submissions -c titanic     # consultar scores
```

(Ejecutadas el 2026-07-08. Recordatorio: máximo 10 al día — por eso el juez de verdad es la CV y el
LB solo confirma.)

## 4 · El veredicto del leaderboard

| Versión | CV local | LB público | Caída CV→LB |
|---|---|---|---|
| Regla del género (referencia) | 0.7868 | 0.7655 | −2.1 pts |
| **v02 Random Forest** | 0.8473 ± 0.009 | **0.76794** | −7.9 pts |
| **v03 Gradient Boosting** | 0.8563 ± 0.024 | **0.74880** | −10.8 pts |

**El ganador de la CV perdió el examen real.** El GB — nuestro campeón por CV — no solo pierde
contra el RF: cae por debajo de la regla del género. Y el RF la supera, pero por poco. Este
resultado vale más que cualquier tutorial; desmenucémoslo:

**a) La advertencia de la Fase 5 era cierta.** Dijimos: *"la ventaja del GB (0.9 pts) es menor que
su propio ruido (±2.4 pts) — no podemos jurarla"*. El fold de 0.899 era suerte, y la parte de
suerte no viaja al test. El RF, con la mitad de media pero un tercio de desviación, generalizó
mejor. **Entre dos modelos de media parecida, elige siempre el de menor varianza.**

**b) El score del ganador del tuning estaba inflado de serie.** El 0.8563 del GB era el mejor de
24 combinaciones × el mejor de 8 modelos — el sesgo de selección del que hablamos: quien gana una
lotería de muchos billetes, en parte gana por azar. La caída de 10.8 puntos es ese azar
devolviéndose.

**c) El LB también tiene su propio ruido.** El test son 418 personas: cada pasajero vale 0.24
puntos. La diferencia RF-género (0.76794 vs 0.76555) es UN pasajero de los 51 en los que nos
jugamos la apuesta. Ni euforia ni drama por diferencias así.

**d) La brecha CV→LB de ~8 puntos NO invalida nuestra CV** — la regla del género (que no aprende
nada) ya caía 2 puntos: parte es que el test simplemente es otra muestra. El resto es sobreajuste
fino al train que la CV no puede detectar del todo. La consecuencia práctica para la Fase 7:
**más robustez y mejores features, no más tuning.**

## 5 · Actualizar el registro

Anotamos los LB reales — el registro queda como memoria oficial del proyecto:

In [4]:
registro = pd.read_csv("../submissions/registro_experimentos.csv")
registro.loc[registro["version"] == "v02-rf-afinado", "lb_publico"] = 0.76794
registro.loc[registro["version"] == "v03-gb-afinado", "lb_publico"] = 0.74880
registro.to_csv("../submissions/registro_experimentos.csv", index=False)
registro

,version,fecha,features,modelo,cv_media,cv_std,lb_publico
0,base-mayoria,2026-07-08,-,DummyClassifier(most_frequent),0.6162,0.0023,NaN
1,base-genero,2026-07-08,IsFemale,ReglaGenero,0.7868,0.0188,0.76550
2,v01-logistica,2026-07-08,16 (Fase 3),StandardScaler+LogisticRegression,0.8328,0.0110,NaN
3,v02-rf-afinado,2026-07-08,16 (Fase 3),"RF(depth=8, leaf=4, feat=0.5)",0.8473,0.0092,0.76794
4,v03-gb-afinado,2026-07-08,16 (Fase 3),"GB(lr=0.03, depth=4, n=100)",0.8563,0.0245,0.74880
5,v04-voting,2026-07-08,16 (Fase 3),Voting soft (log+rf+gb+svm) - DESCARTADO,0.8417,0.0136,NaN


## 6 · Conclusiones de la Fase 6

- Pipeline completo cerrado: datos crudos → limpieza → features → modelo → **submission real en
  Kaggle**. Mejor score: **0.76794 (v02, Random Forest)** — por encima del baseline de género
  (0.7655), por debajo aún de la meta del plan (0.78-0.80).
- Aprendimos a reentrenar con el 100% para el modelo final, validar submissions con asserts,
  usar el CLI de Kaggle, y — lo más valioso — **leer una decepción del leaderboard sin pánico**:
  varianza, sesgo de selección y ruido del test explican cada punto de la caída.
- La apuesta de las features (51 pasajeros contra la regla del género) quedó aproximadamente en
  tablas esta vez. Ahí está el margen de mejora.

**Siguiente — Fase 7 (iteración):** con el diagnóstico de hoy, las líneas más prometedoras son
(1) modelos más simples/robustos — logística y SVM nunca se probaron en el LB y su CV era estable;
(2) features de supervivencia por grupo (familias enteras vivían o morían juntas — el apellido/ticket
lo captura); (3) desempatar GB vs RF vs logística con el LB gastando submissions con cabeza.